# FITWELL · materials: one taxonomy, one classifier
Scans every folder in `datasets/`, labels images with a shared material set, trains a pixelation-robust ResNet18,
and reports confusion + precision@k. Run with **Restart > Run All**. Output: `models/fitwell_materials.pt`.

In [ ]:
# pip install torch torchvision pandas pillow scikit-learn tqdm
from pathlib import Path
import random, numpy as np, pandas as pd, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

EPOCHS = 3               # raise to 6+ once everything runs
MAX_PER_CLASS = 400      # cap per material so CPU training stays fast
IMG_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
random.seed(0); np.random.seed(0); torch.manual_seed(0)

# --- one shared material taxonomy ---
TAXONOMY = {
    "wood": ["wood", "timber", "plywood", "bamboo", "parquet"],
    "brick": ["brick"],
    "concrete": ["concrete", "cement", "asphalt"],
    "stone": ["stone", "granite", "marble", "rock", "slate", "sandstone"],
    "metal": ["metal", "steel", "aluminium", "aluminum", "iron", "copper", "brass"],
    "glass": ["glass", "mirror", "window"],
    "ceramic": ["tile", "ceramic", "porcelain"],
    "plaster": ["plaster", "paint", "stucco", "gypsum"],
    "fabric": ["fabric", "carpet", "cloth", "textile", "wool", "leather"],
    "plastic": ["plastic", "rubber", "vinyl", "laminate"],
    "insulation": ["insulation", "foam", "cork"],
}
def unify(raw):
    r = str(raw).lower()
    for cat, keys in TAXONOMY.items():
        if any(k in r for k in keys):
            return cat
    return "other"

# --- find the datasets folder (searches upward from this notebook) ---
here = Path.cwd()
DATASETS = next((p / "datasets" for p in [here, *here.parents] if (p / "datasets").exists()), None)
assert DATASETS, f"No 'datasets' folder found above {here}"
print("Using datasets:", DATASETS)

# --- scan EVERY dataset folder; label = first folder name in the path that maps to a material ---
rows = []
for ds in sorted(p for p in DATASETS.iterdir() if p.is_dir()):
    n_img = 0
    for p in ds.rglob('*'):
        if p.suffix.lower() in IMG_EXT:
            n_img += 1
            parts = [q for q in reversed(p.relative_to(ds).parts[:-1])]   # nearest folder first
            label = next((unify(q) for q in parts if unify(q) != 'other'), 'other')
            rows.append((str(p), label, ds.name))
    print(f"  {ds.name:<16} images found: {n_img}")
df_all = pd.DataFrame(rows, columns=['path', 'label', 'source'])
df = df_all[df_all.label != 'other'].reset_index(drop=True)
print(f"\nTotal images: {len(df_all)} | with a material label: {len(df)}")
if len(df):
    print(df.groupby(['label', 'source']).size().unstack(fill_value=0))

## Split and loaders

In [ ]:
# keep materials with enough images, cap per class, stratified split
counts = df.label.value_counts()
keep = counts[counts >= 6].index.tolist()
if len(keep) < 2:
    print("Labelled images per material:\n", counts.to_string() if len(counts) else "(none)")
    print("\nNot enough labelled images to train (need >= 2 materials with >= 6 images each).")
    print("Folders are only labelled if a folder name in the image path contains a material word (wood, brick, concrete, ...).")
    print("Add your folder names to TAXONOMY above, or send the table printed in the first cell.")
    raise SystemExit
df = df[df.label.isin(keep)]
df = pd.concat([g.sample(min(len(g), MAX_PER_CLASS), random_state=0) for _, g in df.groupby('label')])
classes = sorted(df.label.unique()); cid = {c: i for i, c in enumerate(classes)}

val_idx = []
for lab, g in df.groupby('label'):
    val_idx += g.sample(n=max(2, int(round(len(g) * .2))), random_state=0).index.tolist()
val = df.loc[val_idx]; tr = df.drop(val_idx)
print(f"classes: {classes}\ntrain images: {len(tr)} | val images: {len(val)}")

class Pixelate:      # makes the net robust to low-resolution wood vs brick
    def __call__(self, im):
        if random.random() < .5:
            f = random.choice([4, 6, 8, 12]); w, h = im.size
            im = im.resize((max(w//f, 4), max(h//f, 4)), Image.BILINEAR).resize((w, h), Image.NEAREST)
        return im
norm = T.Normalize([.485, .456, .406], [.229, .224, .225])
train_tf = T.Compose([T.RandomResizedCrop(160, scale=(.3, 1)), T.RandomHorizontalFlip(), Pixelate(),
                      T.ColorJitter(.3, .3, .2, .02), T.ToTensor(), norm])
val_tf = T.Compose([T.Resize(176), T.CenterCrop(160), T.ToTensor(), norm])

class DS(Dataset):
    def __init__(s, d, tf): s.d, s.tf = d.reset_index(drop=True), tf
    def __len__(s): return len(s.d)
    def __getitem__(s, i):
        for k in range(len(s.d)):                      # skip unreadable images instead of crashing
            r = s.d.iloc[(i + k) % len(s.d)]
            try: return s.tf(Image.open(r.path).convert('RGB')), cid[r.label]
            except Exception: continue
        raise RuntimeError("no readable images")

# num_workers=0: required on Windows + Jupyter
dl = DataLoader(DS(tr, train_tf), batch_size=32, shuffle=True, num_workers=0)
vl = DataLoader(DS(val, val_tf), batch_size=32, num_workers=0)

## Train

In [ ]:
dev = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print("training on:", dev)
net = resnet18(weights=ResNet18_Weights.DEFAULT); net.fc = nn.Linear(512, len(classes)); net.to(dev)
opt = torch.optim.AdamW(net.parameters(), 3e-4, weight_decay=1e-4); lossf = nn.CrossEntropyLoss(label_smoothing=.1)
for ep in range(EPOCHS):
    net.train()
    for x, y in tqdm(dl, desc=f'epoch {ep+1}/{EPOCHS}'):
        x, y = x.to(dev), y.to(dev); opt.zero_grad(); lossf(net(x), y).backward(); opt.step()
    net.eval(); P, Y = [], []
    with torch.no_grad():
        for x, y in vl: P += net(x.to(dev)).argmax(1).cpu().tolist(); Y += y.tolist()
    print('val acc', round(float(np.mean(np.array(P) == np.array(Y))), 3))
Path('../models').mkdir(exist_ok=True)
torch.save({'state': net.state_dict(), 'classes': classes}, '../models/fitwell_materials.pt')
print("saved ../models/fitwell_materials.pt")

## Where does it confuse materials?

In [ ]:
cm = confusion_matrix(Y, P, labels=list(range(len(classes))), normalize='true')
print(pd.DataFrame(cm, index=classes, columns=classes).round(2))
if 'wood' in cid and 'brick' in cid:
    i, j = cid['wood'], cid['brick']
    print('wood->brick', cm[i, j].round(3), '| brick->wood', cm[j, i].round(3))

## Retrieval precision

In [ ]:
net.fc = nn.Identity(); net.eval()
def embed(d):
    out = []
    with torch.no_grad():
        for x, _ in DataLoader(DS(d, val_tf), batch_size=32, num_workers=0): out.append(net(x.to(dev)).cpu())
    return nn.functional.normalize(torch.cat(out), dim=1)
E = embed(val); S = E @ E.T; S.fill_diagonal_(-1)
k = min(5, len(val) - 1)
if k < 1:
    print("Validation set too small for precision@k.")
else:
    top = S.topk(k, dim=1).indices.numpy(); lab = val.label.values
    p = np.array([(lab[top[i]] == lab[i]).mean() for i in range(len(val))])
    print(f'precision@{k} overall:', p.mean().round(3))
    print(pd.Series(p, index=lab).groupby(level=0).mean().round(3))